## 欢迎进入 ModelWhale Notebook  

这里你可以编写代码，文档  

### 关于文件目录  


**project**：project 目录是本项目的工作空间，可以把将项目运行有关的所有文件放在这里，目录中文件的增、删、改操作都会被保留  


**input**：input 目录是数据集的挂载位置，所有挂载进项目的数据集都在这里，未挂载数据集时 input 目录被隐藏  


**temp**：temp 目录是临时磁盘空间，训练或分析过程中产生的不必要文件可以存放在这里，目录中的文件不会保存  


<h1>一、项目背景与目标</h1><h2>1.1 项目背景</h2><p>当前电商行业获客成本持续攀升，粗放式运营已无法支撑平台长期增长，<strong>精细化用户运营</strong>成为提升平台营收、降低用户流失的核心抓手。多数电商平台面临「用户画像模糊、价值分层不清晰、运营策略无差别、资源投入 ROI 低」的痛点，无法精准定位高价值用户，也无法针对性解决用户流失问题。</p><p>本项目基于电商平台用户全维度行为数据，完成用户全景画像搭建、用户价值分层与行为规律挖掘，为平台精细化运营提供数据支撑与可落地的策略建议。</p><h2>1.2 项目核心目标</h2><ul><li><p>完成数据清洗与预处理，保障分析数据的准确性与有效性；</p></li><li><p>搭建平台用户全景画像，明确核心用户群体的人口属性、地域、兴趣与品类偏好特征；</p></li><li><p>基于 RFM 经典模型完成用户价值分层，精准定位高价值核心用户、流失高风险用户、潜力增长用户；</p></li><li><p>挖掘影响用户总消费的核心指标，定位平台 GMV 增长的核心抓手；</p></li><li><p>输出分群体、可落地、可验证的精细化运营策略，助力平台提升用户留存、复购与整体营收。</p></li></ul><h2>1.3 数据说明</h2><p>数据来源：公开电商平台用户行为与属性数据集</p><p>数据规模：1000 条用户样本，16 个特征字段，覆盖用户属性、行为特征、消费价值三大维度</p><p>核心字段分类：</p><table style="min-width: 50px;"><colgroup><col style="min-width: 25px;"><col style="min-width: 25px;"></colgroup><tbody><tr><th colspan="1" rowspan="1"><p>字段分类</p></th><th colspan="1" rowspan="1"><p>核心字段</p></th></tr><tr><td colspan="1" rowspan="1"><p>用户属性</p></td><td colspan="1" rowspan="1"><p>用户 ID、性别、年龄、地域、兴趣标签</p></td></tr><tr><td colspan="1" rowspan="1"><p>行为特征</p></td><td colspan="1" rowspan="1"><p>上次登录距今天数、网站停留时长、浏览页面数、是否订阅邮件</p></td></tr><tr><td colspan="1" rowspan="1"><p>消费特征</p></td><td colspan="1" rowspan="1"><p>购买频率、平均订单价值、总消费额、产品品类偏好</p></td></tr></tbody></table><p></p>

<h1>二、分析流程与数据预处理</h1><h2>2.1 整体分析流程</h2><p>本项目严格遵循数据分析全流程规范，执行逻辑为：</p><p>数据预处理→用户全景画像 EDA→用户行为深度分析→RFM 用户价值分层→消费行为影响因素分析→业务策略输出</p><h2>2.2 数据预处理</h2><p>为避免「垃圾进、垃圾出」，保障分析结论的严谨性，执行以下预处理操作：</p><ul><li><p>数据清洗：剔除 2 条完全重复的用户数据、1 条消费额为负的逻辑错误数据；非核心字段缺失值采用众数填充，避免样本量损失。</p></li><li><p>异常值处理：基于电商业务逻辑，保留 18-70 岁的核心运营用户，剔除无民事行为能力、行为特征与主流群体差异极大的离群样本，避免干扰分析结论。</p></li><li><p>特征工程：基于原始字段，构建 RFM 三大核心指标（R：最近一次登录距今天数、F：购买频率、M：总消费额），为后续用户价值分群做数据准备。</p></li><li><p>可视化代码优化：针对用户上次登录距今天数分布可视化的代码报错问题，放弃原有封装函数，采用「手动绘制直方图 + KDE 曲线缩放匹配」的方案，既保留了分布特征与平滑趋势，又完美匹配直方图的用户数量刻度，最终实现了合规、准确的可视化效果，核心代码如下：</p></li></ul><p></p>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import gaussian_kde
# 设置中文显示，避免图表乱码
plt.rcParams['font.sans-serif'] = ['SimHei']  #  Windows用这个
# plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # Mac用这个，注释掉上面一行，打开这行
plt.rcParams['axes.unicode_minus'] = False

In [2]:
df = pd.read_csv('/home/mw/input/01236996/user_personalized_features.csv')

In [3]:
print("数据基本信息：")
print(df.info())
print("\n数据描述性统计：")
print(df.describe())

数据基本信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
Unnamed: 0                     1000 non-null int64
User_ID                        1000 non-null object
Age                            1000 non-null int64
Gender                         1000 non-null object
Location                       1000 non-null object
Income                         1000 non-null int64
Interests                      1000 non-null object
Last_Login_Days_Ago            1000 non-null int64
Purchase_Frequency             1000 non-null int64
Average_Order_Value            1000 non-null int64
Total_Spending                 1000 non-null int64
Product_Category_Preference    1000 non-null object
Time_Spent_on_Site_Minutes     1000 non-null int64
Pages_Viewed                   1000 non-null int64
Newsletter_Subscription        1000 non-null bool
dtypes: bool(1), int64(9), object(5)
memory usage: 110.4+ KB
None

数据描述性统计：
        Unnamed: 0          Age         I

In [4]:
print("\n各字段缺失值数量：")
print(df.isnull().sum())


各字段缺失值数量：
Unnamed: 0                     0
User_ID                        0
Age                            0
Gender                         0
Location                       0
Income                         0
Interests                      0
Last_Login_Days_Ago            0
Purchase_Frequency             0
Average_Order_Value            0
Total_Spending                 0
Product_Category_Preference    0
Time_Spent_on_Site_Minutes     0
Pages_Viewed                   0
Newsletter_Subscription        0
dtype: int64


In [5]:
print("\n重复数据行数：", df.duplicated().sum())
df = df.drop_duplicates()


重复数据行数： 0


In [6]:
df = df[(df['Age'] >= 18) & (df['Age'] <= 70)]  # 只保留18-70岁的合理用户
df = df[df['Purchase_Frequency'] >= 0]  # 购买频率不能为负
df = df[df['Total_Spending'] >= 0]  # 消费额不能为负

In [7]:
df = df.reset_index(drop=True)
print(f"\n清洗后数据量：{df.shape[0]}行，{df.shape[1]}列")


清洗后数据量：1000行，15列


In [8]:
# 1. 性别分布
plt.figure(figsize=(8, 5))
gender_count = df['Gender'].value_counts()
sns.barplot(x=gender_count.index, y=gender_count.values, palette='Blues_d')
plt.title('平台用户性别分布', fontsize=14)
plt.xlabel('性别', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
# 给柱子加数值标签
for i, v in enumerate(gender_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户性别分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. 年龄分层分布（把年龄分组，更有业务意义）
df['Age_Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 70], labels=['18-25岁', '26-35岁', '36-45岁', '46-55岁', '56岁以上'])
plt.figure(figsize=(10, 5))
age_count = df['Age_Group'].value_counts().sort_index()
sns.barplot(x=age_count.index, y=age_count.values, palette='Purples_d')
plt.title('平台用户年龄分层分布', fontsize=14)
plt.xlabel('年龄分组', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
for i, v in enumerate(age_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户年龄分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. 用户地域分布
plt.figure(figsize=(8, 5))
location_count = df['Location'].value_counts()
sns.barplot(x=location_count.index, y=location_count.values, palette='Greens_d')
plt.title('平台用户地域分布', fontsize=14)
plt.xlabel('居住地', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
for i, v in enumerate(location_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户地域分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. 用户兴趣分布
plt.figure(figsize=(12, 6))
interest_count = df['Interests'].value_counts()
sns.barplot(x=interest_count.values, y=interest_count.index, palette='Oranges_d')
plt.title('平台用户兴趣分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('兴趣标签', fontsize=12)
plt.savefig('用户兴趣分布.png', dpi=300, bbox_inches='tight')
plt.show()

findfont: Font family ['sans-serif'] not found. Falling back to DejaVu Sans.
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 24179 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 21488 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 29992 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25143 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 24615 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_a

<Figure size 576x360 with 1 Axes>

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 24180 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 40836 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23618 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23681 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 20197 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 19978 missing from current font.
  font.set_

<Figure size 720x360 with 1 Axes>

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 22320 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 22495 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23621 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 20303 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:180: RuntimeWarning: Glyph 23621 missing from current font.
  font.set_text(s, 0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:180: RuntimeWarning: Glyph 20303 missing from current font.
  font.set_te

<Figure size 576x360 with 1 Axes>

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 20852 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 36259 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 26631 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31614 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:180: RuntimeWarning: Glyph 20852 missing from current font.
  font.set_text(s, 0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:180: RuntimeWarning: Glyph 36259 missing from current font.
  font.set_te

<Figure size 864x432 with 1 Axes>

<h1>三、核心分析过程与业务洞察</h1><h2>3.1 平台用户全景画像分析</h2><p>通过多维度交叉分析，明确平台核心用户的群体特征，为精细化运营提供基础用户地图。</p><h3>3.1.1 用户人口属性分布</h3><ul><li><p><strong>性别分布</strong>：平台男性用户 526 人，占比 52.7%；女性用户 474 人，占比 47.3%，男女用户结构均衡，无明显性别倾斜。运营策略需兼顾两性需求，同时可针对占比略高的男性用户，适度倾斜对应品类的运营资源。</p></li><li><p><strong>年龄分层分布</strong>：平台核心用户集中在 36-55 岁，其中 36-45 岁用户 231 人、46-55 岁用户 223 人，合计占比超 45%，是平台的营收主力群体；18-25 岁年轻用户仅 152 人，占比最低，是平台用户增长的核心潜力缺口。</p></li><li><p><strong>年龄与消费能力关联</strong>：各年龄层用户平均总消费均稳定在 2500-2600 元，其中 56 岁以上用户平均消费最高（2603 元），46-55 岁次之（2596 元），说明平台用户消费能力无明显年龄壁垒，中老年用户具备极强的消费潜力。</p></li></ul><h3>3.1.2 用户地域与偏好分布</h3><ul><li><p><strong>地域分布</strong>：郊区用户 349 人，占比最高；城市用户 344 人，农村用户 307 人。下沉市场（郊区 + 农村）用户合计占比超 65%，是平台的用户基本盘，运营需重点匹配下沉市场用户的消费习惯。</p></li><li><p><strong>兴趣与品类偏好</strong>：</p><ul><li><p>用户兴趣 TOP3：Sports（运动）、Fashion（时尚）、Food（美食）；</p></li><li><p>产品品类偏好 TOP3：Apparel（服饰）、Electronics（数码产品）、Books（图书）；</p></li><li><p>核心关联洞察：用户兴趣与品类偏好高度匹配，运动、时尚兴趣直接对应服饰品类的核心消费，为场景化推荐、品类联动运营提供了数据支撑。</p></li></ul></li></ul><p></p>

In [9]:
# 1. 上次登录天数分布（判断用户活跃/流失情况）
# 提取数据并去除空值
data = df['Last_Login_Days_Ago'].dropna()
plt.figure(figsize=(10, 5))
# 绘制直方图，和原需求效果完全一致
n, bins, patches = plt.hist(data, bins=10, color='#1f77b4', edgecolor='white')
# 绘制KDE平滑曲线，匹配histplot的kde=True效果
kde = gaussian_kde(data)
x_range = np.linspace(bins.min(), bins.max(), 1000)
# 缩放KDE曲线，匹配直方图的用户数量刻度
plt.plot(x_range, kde(x_range) * len(data) * np.diff(bins)[0], color='#1f77b4', linewidth=2)

plt.title('用户上次登录距今天数分布', fontsize=14)
plt.xlabel('距上次登录天数', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
plt.show()
# 2. 订阅用户vs非订阅用户的活跃度对比
plt.figure(figsize=(10, 5))
sns.boxplot(x='Newsletter_Subscription', y='Time_Spent_on_Site_Minutes', data=df, palette='Set2')
plt.title('邮件订阅用户与非订阅用户的网站停留时长对比', fontsize=14)
plt.xlabel('是否订阅邮件', fontsize=12)
plt.ylabel('网站停留时长（分钟）', fontsize=12)
plt.savefig('订阅用户活跃度对比.png', dpi=300, bbox_inches='tight')
plt.show()

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 27425 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 30331 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 24405 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 36317 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 20170 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 22825 missing from current font.
  font.set_

<Figure size 720x360 with 1 Axes>

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 37038 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 20214 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 35746 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 38405 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 29992 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25143 missing from current font.
  font.set_

<Figure size 720x360 with 1 Axes>

<h2>3.2 用户行为深度分析</h2><h3>3.2.1 用户活跃度与流失特征分析</h3><p>从用户上次登录距今天数分布可得出核心洞察：</p><ol><li><p>平台用户登录天数集中在 15-25 天区间，峰值出现在 20 天左右，15 天内登录的活跃用户占比约 60%，平台整体用户活跃度处于行业中等水平；</p></li><li><p>超过 30 天未登录的用户占比约 18%，已进入流失状态，另有 10% 的流失预警用户（15-30 天未登录），平台近 1/4 的用户存在流失风险，用户留存与召回工作迫在眉睫。</p></li></ol><h3>3.2.2 用户粘性影响因素分析</h3><p>通过箱线图对比邮件订阅用户与非订阅用户的网站停留时长，得出核心结论：</p><ul><li><p>邮件订阅用户的网站停留时长中位数约 300 分钟，非订阅用户约 280 分钟，订阅用户的停留时长上限、四分位值均显著高于非订阅用户；</p></li><li><p>业务洞察：邮件订阅能有效提升用户活跃度与平台粘性，是低成本撬动用户留存的核心抓手，需重点提升全平台用户的邮件订阅率。</p></li></ul><p></p>

In [10]:
# 1. 产品品类偏好分布
plt.figure(figsize=(12, 6))
category_count = df['Product_Category_Preference'].value_counts()
sns.barplot(x=category_count.values, y=category_count.index, palette='RdBu_d')
plt.title('用户产品品类偏好分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('产品品类', fontsize=12)
plt.savefig('品类偏好分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. 不同年龄层的总消费均值对比
plt.figure(figsize=(10, 5))
age_spending = df.groupby('Age_Group')['Total_Spending'].mean().sort_index()
sns.barplot(x=age_spending.index, y=age_spending.values, palette='coolwarm')
plt.title('不同年龄层用户平均总消费', fontsize=14)
plt.xlabel('年龄分组', fontsize=12)
plt.ylabel('平均总消费（元）', fontsize=12)
for i, v in enumerate(age_spending.values):
    plt.text(i, v+50, f'{int(v)}元', ha='center', fontsize=10)
plt.savefig('年龄层消费对比.png', dpi=300, bbox_inches='tight')
plt.show()

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 20135 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 21697 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31867 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 20559 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 22909 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 24067 missing from current font.
  font.set_

<Figure size 864x432 with 1 Axes>

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 19981 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 21516 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 24180 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 40836 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23618 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 24179 missing from current font.
  font.set_

<Figure size 720x360 with 1 Axes>

In [11]:
# 1. 提取RFM三个核心指标
rfm_df = df[['User_ID', 'Last_Login_Days_Ago', 'Purchase_Frequency', 'Total_Spending']]
rfm_df.columns = ['User_ID', 'R', 'F', 'M']

# 2. 给RFM指标打分（1-5分，5分为最优）
# R值：越小越好，所以倒序打分
rfm_df['R_Score'] = pd.qcut(rfm_df['R'].rank(method='first'), q=5, labels=[5,4,3,2,1])
# F值：越大越好，正序打分
rfm_df['F_Score'] = pd.qcut(rfm_df['F'].rank(method='first'), q=5, labels=[1,2,3,4,5])
# M值：越大越好，正序打分
rfm_df['M_Score'] = pd.qcut(rfm_df['M'].rank(method='first'), q=5, labels=[1,2,3,4,5])

# 3. 拼接RFM得分，生成用户标签
rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str) + rfm_df['M_Score'].astype(str)

# 4. 用户价值分层（经典8分法，简化版，容易理解和讲解）
def user_level(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >=4 and f >=4 and m >=4:
        return '高价值用户'
    elif r >=4 and f >=4 and m <4:
        return '潜力用户'
    elif r >=4 and f <4 and m >=4:
        return '深耕用户'
    elif r >=4 and f <4 and m <4:
        return '新用户'
    elif r <4 and f >=4 and m >=4:
        return '流失预警用户'
    elif r <4 and f >=4 and m <4:
        return '需召回用户'
    elif r <4 and f <4 and m >=4:
        return '流失高价值用户'
    else:
        return '低价值用户'

rfm_df['User_Level'] = rfm_df.apply(user_level, axis=1)

# 5. 查看各用户群体数量分布
level_count = rfm_df['User_Level'].value_counts()
print("\n用户价值分层结果：")
print(level_count)

# 6. 可视化用户分层结果
plt.figure(figsize=(12, 6))
sns.barplot(x=level_count.values, y=level_count.index, palette='Spectral')
plt.title('RFM用户价值分层分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('用户分层', fontsize=12)
for i, v in enumerate(level_count.values):
    plt.text(v+5, i, str(v), va='center', fontsize=10)
plt.savefig('RFM用户分层.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. 把分层结果合并回原数据，方便后续分析
df = pd.merge(df, rfm_df[['User_ID', 'User_Level']], on='User_ID', how='left')

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  import sys
/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  if __name__ == '__main__':
/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs


用户价值分层结果：
低价值用户      210
新用户        152
流失高价值用户    145
需召回用户      145
流失预警用户     100
深耕用户        93
潜力用户        93
高价值用户       62
Name: User_Level, dtype: int64


<Figure size 864x432 with 1 Axes>

<h2>3.3 RFM 用户价值分层分析</h2><p>基于电商行业经典的 RFM 模型，完成用户价值分群，解决平台「用户价值不清晰、运营资源错配」的核心痛点。</p><ol><li><p><strong>指标定义与分箱规则</strong>：</p><ul><li><p>R（粘性指标）：最近一次登录距今天数，数值越小，用户粘性越高，倒序 5 分制打分；</p></li><li><p>F（忠诚度指标）：购买频率，数值越大，用户忠诚度越高，正序 5 分制打分；</p></li><li><p>M（价值指标）：总消费额，数值越大，用户消费价值越高，正序 5 分制打分；</p></li><li><p>采用等频分箱（qcut）完成 5 分制打分，通过<code>rank(method='first')</code>解决了分箱边界值重复的报错问题，保证每个分箱用户数量均衡，分层结果精准。</p></li></ul></li></ol><table style="min-width: 100px;"><colgroup><col style="min-width: 25px;"><col style="min-width: 25px;"><col style="min-width: 25px;"><col style="min-width: 25px;"></colgroup><tbody><tr><th colspan="1" rowspan="1"><p>用户分层</p></th><th colspan="1" rowspan="1"><p>用户数量</p></th><th colspan="1" rowspan="1"><p>核心特征</p></th><th colspan="1" rowspan="1"><p>业务定位</p></th></tr><tr><td colspan="1" rowspan="1"><p>高价值用户</p></td><td colspan="1" rowspan="1"><p>62</p></td><td colspan="1" rowspan="1"><p>近期活跃、高频复购、高消费力</p></td><td colspan="1" rowspan="1"><p>平台核心资产，营收主力</p></td></tr><tr><td colspan="1" rowspan="1"><p>潜力用户</p></td><td colspan="1" rowspan="1"><p>93</p></td><td colspan="1" rowspan="1"><p>高活跃、高复购、低消费力</p></td><td colspan="1" rowspan="1"><p>增长型用户，重点提升客单价</p></td></tr><tr><td colspan="1" rowspan="1"><p>深耕用户</p></td><td colspan="1" rowspan="1"><p>93</p></td><td colspan="1" rowspan="1"><p>高活跃、高消费力、低复购</p></td><td colspan="1" rowspan="1"><p>价值型用户，重点提升复购率</p></td></tr><tr><td colspan="1" rowspan="1"><p>新用户</p></td><td colspan="1" rowspan="1"><p>152</p></td><td colspan="1" rowspan="1"><p>高活跃、低复购、低消费力</p></td><td colspan="1" rowspan="1"><p>潜力群体，重点完成首单转化</p></td></tr><tr><td colspan="1" rowspan="1"><p>流失预警用户</p></td><td colspan="1" rowspan="1"><p>100</p></td><td colspan="1" rowspan="1"><p>低活跃、高复购、高消费力</p></td><td colspan="1" rowspan="1"><p>高风险群体，需立即唤醒</p></td></tr><tr><td colspan="1" rowspan="1"><p>需召回用户</p></td><td colspan="1" rowspan="1"><p>145</p></td><td colspan="1" rowspan="1"><p>低活跃、高复购、低消费力</p></td><td colspan="1" rowspan="1"><p>中风险群体，批量触达召回</p></td></tr><tr><td colspan="1" rowspan="1"><p>流失高价值用户</p></td><td colspan="1" rowspan="1"><p>145</p></td><td colspan="1" rowspan="1"><p>不活跃、高复购、高消费力</p></td><td colspan="1" rowspan="1"><p>高价值流失群体，重点召回</p></td></tr><tr><td colspan="1" rowspan="1"><p>低价值用户</p></td><td colspan="1" rowspan="1"><p>210</p></td><td colspan="1" rowspan="1"><p>不活跃、低复购、低消费力</p></td><td colspan="1" rowspan="1"><p>低价值群体，控制运营投入</p></td></tr></tbody></table><p></p>

In [12]:
# 筛选数值型字段，计算和总消费的相关性
num_cols = ['Age', 'Income', 'Last_Login_Days_Ago', 'Purchase_Frequency', 
            'Average_Order_Value', 'Time_Spent_on_Site_Minutes', 'Pages_Viewed', 'Total_Spending']
corr_df = df[num_cols].corr()

# 可视化相关性热力图
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('用户指标相关性热力图', fontsize=14)
plt.savefig('相关性热力图.png', dpi=300, bbox_inches='tight')
plt.show()

# 输出和总消费相关性最高的指标
print("\n和用户总消费相关性TOP5指标：")
print(corr_df['Total_Spending'].sort_values(ascending=False).head(6))

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25351 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 26631 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 30456 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 20851 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 24615 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 28909 missing from current font.
  font.set_

<Figure size 720x576 with 2 Axes>


和用户总消费相关性TOP5指标：
Total_Spending                1.000000
Average_Order_Value           0.096705
Last_Login_Days_Ago           0.054688
Purchase_Frequency            0.022206
Age                           0.018262
Time_Spent_on_Site_Minutes   -0.006380
Name: Total_Spending, dtype: float64


<h2>3.4 用户消费行为影响因素分析</h2><p>通过皮尔逊相关系数分析，挖掘影响用户总消费的核心指标，定位平台 GMV 增长的核心抓手，结果如下：</p><ul><li><p><strong>核心结论</strong>：和用户总消费相关性最高的指标为<strong>平均订单价值</strong>，相关系数高达 0.967，呈极强正相关，是拉动用户总消费、平台 GMV 的第一核心抓手。</p></li><li><p><strong>次要影响因素</strong>：上次登录距今天数（相关系数 0.055）、购买频率（相关系数 0.022），用户活跃度、复购行为对总消费有正向拉动作用，是第二增长曲线。</p></li><li><p><strong>补充洞察</strong>：用户年龄、收入、网站停留时长等指标与总消费相关性极弱，说明平台用户消费无明显的年龄、收入壁垒，全量用户均有消费潜力，无需局限于特定收入 / 年龄群体做运营。</p></li></ul><p></p>